In [ ]:
pip install osmnx

In [ ]:
import folium
import osmnx as ox
import networkx as nx
import random

In [ ]:
G = ox.graph_from_place('Quận 10, Hồ Chí Minh, Việt Nam', network_type='drive')
m = folium.Map(location=[10.765, 106.675], zoom_start=13)
customers, drivers = 5, 10
customers_nodes = random.sample(list(G.nodes()), customers)
drivers_nodes = random.sample(list(G.nodes()), drivers)

In [ ]:
for node in customers_nodes:
  folium.Marker(
      location = [G.nodes[node]['y'], G.nodes[node]['x']],
      icon = folium.Icon(color='blue'),
      popup='Customer'
  ).add_to(m)

for node in drivers_nodes:
  folium.Marker(
      location = [G.nodes[node]['y'], G.nodes[node]['x']],
      icon = folium.Icon(color='red'),
      popup='Driver'
  ).add_to(m)

In [ ]:
def match_rides(G, drivers_nodes, customers_nodes):
  match = []
  available_drivers = drivers_nodes.copy()
  for cus in customers_nodes:
    nearest_drivers = None
    min_distance = float('inf')
    for dri in available_drivers:
      distance = nx.shortest_path_length(G, cus, dri, weight='length')
      if distance < min_distance:
        min_distance = distance
        nearest_drivers = dri
    if nearest_drivers is not None:
      match.append((nearest_drivers, cus, min_distance))
      available_drivers.remove(nearest_drivers)
  return match

In [ ]:
for dri, cus, distance in match_rides(G, drivers_nodes, customers_nodes):
  route = nx.shortest_path(G, dri, cus, weight='length')
  folium.PolyLine(
      locations=[(G.nodes[node]['y'], G.nodes[node]['x']) for node in route],
      color='green'
  ).add_to(m)
m